# Lucky Plus | August push uplift

The Sunday CVM request is third in triage. The lead wants to know whether the August 15 push increased subscriptions and whether the result supports a full-base rollout. The comparison uses the randomized treatment and holdout assignments supplied in the extract. All timestamps are interpreted as UTC, and subscription starts are date-only.

In [1]:
from math import comb, sqrt
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "data").is_dir())
DATA = ROOT / "data"
CAMPAIGN = "plus_push_aug"
LAUNCH_DAY = pd.Timestamp("2026-08-15")
EXTRACT_DATE = pd.Timestamp("2026-11-05")
GROUPS = ["treatment", "holdout"]


def read_table(table, dates=()):
    path, = DATA.glob(f"*{table}.csv")
    return pd.read_csv(path, parse_dates=list(dates))


users = read_table("users")
raw_txn = read_table("transactions", ["txn_ts"])
assignments = read_table("cvm_campaigns", ["sent_ts"])
subscriptions = read_table("plus_subscriptions", ["plus_start_date", "plus_end_date"])

campaign = assignments.loc[assignments.campaign_id.eq(CAMPAIGN)].copy()
assert set(campaign.group) == set(GROUPS)
assert campaign.user_id.notna().all() and campaign.user_id.is_unique
assert campaign.user_id.isin(users.user_id).all()
assert subscriptions.plus_start_date.notna().all()
assert not subscriptions.duplicated(["user_id", "plus_start_date"]).any()
assert subscriptions.user_id.isin(users.user_id).all()

qa_names = {"qa lucky", "test user", "test qa", "test test", "تجربه تطبيق", "تجربة"}
qa_ids = users.loc[users.full_name.str.strip().str.casefold().isin(qa_names), "user_id"]
assert not campaign.user_id.isin(qa_ids).any(), "Review QA accounts in campaign assignments"

treatment = campaign.group.eq("treatment")
assert campaign.sent_ts.notna().eq(treatment).all()
assert campaign.loc[treatment, "sent_ts"].dt.normalize().eq(LAUNCH_DAY).all()
first_send = campaign.loc[treatment, "sent_ts"].min()

## Eligibility and assignment checks

The dataset README defines campaign eligibility as at least three successful transactions before send. Use the first observed send as a common cutoff for both groups, since holdout users have no send timestamp. The eligibility check describes the assigned cohort. It does not reassign or filter users for the primary estimate.

In [2]:
txn = raw_txn.drop_duplicates()
assert txn.txn_id.is_unique, "Review conflicting transaction ID records"
prior_counts = txn.loc[txn.status.eq("success") & txn.txn_ts.lt(first_send)].groupby("user_id").size()
campaign["prior_successes"] = campaign.user_id.map(prior_counts).fillna(0).astype(int)
campaign["eligible_at_first_send"] = campaign.prior_successes.ge(3)

active_before = subscriptions.plus_start_date.lt(LAUNCH_DAY) & (
    subscriptions.plus_end_date.isna() | subscriptions.plus_end_date.gt(LAUNCH_DAY)
)
prior_subscribers = subscriptions.loc[active_before, "user_id"]
campaign["active_before_launch"] = campaign.user_id.isin(prior_subscribers)

audit = campaign.groupby("group").agg(
    assigned_users=("user_id", "size"),
    has_send_timestamp=("sent_ts", "count"),
    eligible_at_first_send=("eligible_at_first_send", "sum"),
    already_active_plus=("active_before_launch", "sum"),
    median_prior_successes=("prior_successes", "median"),
).reindex(GROUPS)
print(f"Common eligibility cutoff: {first_send} UTC")
display(audit)

Common eligibility cutoff: 2026-08-15 10:00:00 UTC


,assigned_users,has_send_timestamp,eligible_at_first_send,already_active_plus,median_prior_successes
group,,,,,
treatment,560,560,560,0,6.0
holdout,137,0,137,0,6.0


## Subscription effect

Primary outcome: at least one Plus start during the 30 calendar days after launch (August 16 through September 14 inclusive). Each assigned user stays in their original group and counts once. August 15 starts are ambiguous because the subscription table lacks the time of day, so the primary window starts the next day. The 30-day horizon was chosen for this analysis. Shorter windows and a same-date sensitivity check show how the result depends on that choice.

The primary effect is the treatment rate minus the holdout rate, in percentage points. The notebook also gives the rate ratio when the holdout rate is positive, the estimated incremental starts among assigned treatment users, a 95% Newcombe score interval for the rate difference, and a two-sided Fisher exact test for the primary comparison. The interval reflects sampling uncertainty within this campaign cohort.

In [3]:
def conversion_table(start, end, cohort=campaign):
    assert end <= EXTRACT_DATE
    started_ids = subscriptions.loc[subscriptions.plus_start_date.between(start, end), "user_id"]
    outcome = cohort.assign(converted=cohort.user_id.isin(started_ids))
    summary = outcome.groupby("group").converted.agg(assigned_users="size", conversions="sum").reindex(GROUPS)
    summary["conversion_rate_pct"] = 100 * summary.conversions / summary.assigned_users
    return summary


def wilson_interval(conversions, total):
    z = 1.959963984540054
    rate = conversions / total
    denominator = 1 + z**2 / total
    center = (rate + z**2 / (2 * total)) / denominator
    margin = z * sqrt(rate * (1 - rate) / total + z**2 / (4 * total**2)) / denominator
    return center - margin, center + margin


def compare(summary):
    t, h = summary.loc["treatment"], summary.loc["holdout"]
    nt, nh = int(t.assigned_users), int(h.assigned_users)
    xt, xh = int(t.conversions), int(h.conversions)
    pt, ph = xt / nt, xh / nh
    lt, ut = wilson_interval(xt, nt)
    lh, uh = wilson_interval(xh, nh)
    difference = pt - ph
    return {
        "lift_pp": 100 * difference,
        "ci95_low_pp": 100 * (difference - sqrt((pt - lt)**2 + (uh - ph)**2)),
        "ci95_high_pp": 100 * (difference + sqrt((ut - pt)**2 + (ph - lh)**2)),
        "rate_ratio": pt / ph if ph else float("nan"),
        "incremental_starts_in_treatment": nt * difference,
    }


def fisher_two_sided(summary):
    # With margins fixed, weight each feasible treatment conversion count.
    nt, nh = map(int, summary.assigned_users)
    xt, xh = map(int, summary.conversions)
    total = xt + xh
    observed_weight = comb(nt, xt) * comb(nh, xh)
    weights = (
        comb(nt, x) * comb(nh, total - x)
        for x in range(max(0, total - nh), min(nt, total) + 1)
    )
    return sum(w for w in weights if w <= observed_weight) / comb(nt + nh, total)


first_day = LAUNCH_DAY + pd.Timedelta(days=1)
last_day = LAUNCH_DAY + pd.Timedelta(days=30)
primary = conversion_table(first_day, last_day)
primary_effect = compare(primary)
primary_p = fisher_two_sided(primary)

print(f"Primary window: {first_day:%Y-%m-%d} through {last_day:%Y-%m-%d}")
display(primary.round(3))
display(pd.Series({**primary_effect, "fisher_two_sided_p": primary_p}, name="estimate").to_frame())

Primary window: 2026-08-16 through 2026-09-14


,assigned_users,conversions,conversion_rate_pct
group,,,
treatment,560,71,12.679
holdout,137,9,6.569


,estimate
lift_pp,6.109228
ci95_low_pp,0.116878
ci95_high_pp,10.413922
rate_ratio,1.929960
incremental_starts_in_treatment,34.211679
fisher_two_sided_p,0.051052


In [4]:
windows = [
    ("7 days after launch", first_day, LAUNCH_DAY + pd.Timedelta(days=7)),
    ("14 days after launch", first_day, LAUNCH_DAY + pd.Timedelta(days=14)),
    ("30 days after launch (primary)", first_day, last_day),
    ("Include ambiguous launch-day starts", LAUNCH_DAY, last_day),
]
window_rows = []
for name, start, end in windows:
    summary = conversion_table(start, end)
    window_rows.append({
        "window": name, "start": start.date(), "end": end.date(),
        "treatment_starts": summary.loc["treatment", "conversions"],
        "holdout_starts": summary.loc["holdout", "conversions"],
        **compare(summary),
    })
window_results = pd.DataFrame(window_rows).set_index("window")
display(window_results.round(3))

if campaign.active_before_launch.any():
    print("Secondary comparison among users without an active Plus subscription before launch")
    display(conversion_table(first_day, last_day, campaign.loc[~campaign.active_before_launch]).round(3))

,start,end,treatment_starts,holdout_starts,lift_pp,ci95_low_pp,ci95_high_pp,rate_ratio,incremental_starts_in_treatment
window,,,,,,,,,
7 days after launch,2026-08-16,2026-08-22,14,3,0.310,-3.863,2.503,1.142,1.737
14 days after launch,2026-08-16,2026-08-29,29,5,1.529,-3.334,4.528,1.419,8.562
30 days after launch (primary),2026-08-16,2026-09-14,71,9,6.109,0.117,10.414,1.930,34.212
Include ambiguous launch-day starts,2026-08-15,2026-09-14,72,9,6.288,0.288,10.603,1.957,35.212


## Recommendation and context for the final write-up

Within 30 days after launch, **71/560 treatment users (12.68%)** and **9/137 holdout users (6.57%)** started Plus. The estimated lift is **+6.11 percentage points** (95% CI **+0.12 to +10.41 points**). Keep the conversion counts alongside any rate ratio because a small holdout conversion count makes the multiplier sensitive to one additional start.

Use the result to guide rollout among users who meet the original activity rule. **Keep the expansion within that eligible segment and retain a randomized holdout.** A test in lower-activity segments would be needed to estimate the effect there before sending to the full base. Track absolute lift and incremental starts over a common window.

This section records the evidence and rollout rationale for the later case deliverables.